# Decoding likelihood in MT/MST

Walker et al. (2020) style likelihood decoders on the hanzo wagering sessions.
For every session: count spikes 80–330 ms after motion onset, decode signed coherence with
four decoders (fixed width, rank 1, rank 2, full), then ask how well each decoder's likelihood
predicts the monkey's choice and wager.

**How to use.** Run top to bottom. The decoding step is slow, but each session is saved in `results/`,
so rerunning it only does sessions that aren't saved yet. Everything after it (behaviour, figures,
numbers) runs from the saved results in seconds, so those cells can be edited and rerun freely.
Delete `results/*.pkl` to redo the decoders from scratch.

Figures (saved in `figures/`): 1 firing over time, 2 uncertainty, 3 loadings, 4 decoding, 5 predictive power.

In [ ]:
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import ttest_1samp
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

DATA = Path("/home/cfetsch1/vast-cfetsch1/data/hanzo_neuro_binary")
FS = 30000                      # samples per second
WINDOW = (0.08, 0.33)           # seconds after motion onset
MIN_UNITS, MIN_TRIALS = 5, 200
MODELS = ["fixed width", "rank 1", "rank 2", "full"]

Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

## Loading a session

Trials come from the task's text messages, motion on/off from TTL channel 4. Only completed
left/right `Dots_PDW` trials are kept. Coherence is signed, rightward positive. Spike counts are taken
in the window, and each unit's slow drift over the session is removed (a cubic fit to what's left after
subtracting each coherence's mean).

In [ ]:
def find(folder, pattern):
    hits = sorted(Path(folder).glob(pattern))
    if len(hits) != 1:
        raise ValueError(f"expected one {pattern} in {folder}")
    return hits[0]


def read_trials(rec):
    """One row per trial from the task's text messages, plus motion on/off times."""
    folder = find(rec, "events/Network_Events-*/TEXT_group_1")
    times = np.load(folder / "timestamps.npy")
    texts = np.load(folder / "text.npy", mmap_mode="r")

    rows, block, trial = [], None, None
    for t, text in zip(times, texts):
        key, _, value = text.decode(errors="replace").strip().partition(" ")
        if key in ["Dots_PDW", "mappingMT", "mappingLIP"]:
            block = key
        elif key == "TrialStart":
            trial = {"trial_number": int(value), "start_time": t, "block": block}
            rows.append(trial)
        elif trial is None or key == "IsRecording":
            continue
        elif value:
            trial[key] = value
        else:
            trial[key + "_time"] = t
    trials = pd.DataFrame(rows)

    ttl = find(rec, "events/Rhythm_FPGA-*/TTL_1")
    ttl_times = np.load(ttl / "timestamps.npy")
    states = np.load(ttl / "channel_states.npy")
    ons, offs = ttl_times[states == 4], ttl_times[states == -4]

    starts = trials["start_time"].to_numpy()
    next_starts = np.append(starts[1:], np.iinfo(np.int64).max)
    trials["motion_on"], trials["motion_off"] = -1, -1
    for i, (start, end) in enumerate(zip(starts, next_starts)):
        j = np.searchsorted(ons, start)
        if j < len(ons) and ons[j] < end:
            k = np.searchsorted(offs, ons[j])
            trials.loc[i, "motion_on"] = ons[j]
            trials.loc[i, "motion_off"] = offs[k] if k < len(offs) else -1
    return trials


def load_task(session, rec):
    """Completed left/right wagering trials. Parsing is slow, so it's cached."""
    cache = Path("results") / f"{session.name}_trials.csv"
    if cache.exists():
        trials = pd.read_csv(cache, dtype={"GoodOrBadTrial": str})
    else:
        trials = read_trials(rec)
        trials.to_csv(cache, index=False)

    direction = trials["Direction"].astype(float)
    done = (trials["block"].eq("Dots_PDW") & trials["choice"].notna() & trials["PDW"].notna()
            & trials["GoodOrBadTrial"].astype(str).eq("1")
            & trials["motion_on"].ge(0) & trials["motion_off"].ge(0)
            & direction.isin([0, 180]))
    task = trials[done].copy()

    sign = np.where(task["Direction"].astype(float) == 0, 1, -1)
    task["coherence"] = np.round(task["Coherence"].astype(float) * sign, 4) + 0.0
    task["choice"] = np.where(task["choice"].astype(int) == 2, 1, -1)      # right = 1
    task["wager"] = (task["PDW"].astype(int) == 1).astype(int)             # high = 1
    return task


def load_units(rec):
    """Spike times of every MT/MST unit that isn't labelled noise."""
    cont = find(rec, "continuous/Rhythm_FPGA-*")
    info = pd.read_csv(cont / "kilosort" / "cluster_info.tsv", sep="\t")
    if "group" not in info.columns:
        info["group"] = np.nan
    units = info[info["Region"].isin(["MT", "MST"]) & (info["group"] != "noise")]

    clock_start = int(np.load(cont / "timestamps.npy", mmap_mode="r")[0])
    spike_times = np.load(cont / "kilosort" / "spike_times.npy").ravel().astype(np.int64) + clock_start
    spike_units = np.load(cont / "kilosort" / "spike_clusters.npy").ravel()
    return [np.sort(spike_times[spike_units == u]) for u in units["id"]]


def spike_counts(task, spikes):
    """Counts in the window, with each unit's slow drift removed."""
    start = task["motion_on"].to_numpy() + round(WINDOW[0] * FS)
    end = start + round((WINDOW[1] - WINDOW[0]) * FS)
    keep = end <= task["motion_off"].to_numpy()
    task, start, end = task[keep], start[keep], end[keep]

    counts = np.column_stack([np.searchsorted(s, end) - np.searchsorted(s, start) for s in spikes]).astype(float)
    coh = task["coherence"].to_numpy(dtype=float)

    residual = counts.copy()
    for c in np.unique(coh):
        residual[coh == c] -= counts[coh == c].mean(axis=0)
    x = np.linspace(-1, 1, len(counts))
    for u in range(counts.shape[1]):
        trend = np.polyval(np.polyfit(x, residual[:, u], 3), x)
        counts[:, u] -= trend - trend.mean()
    return task, counts

## Decoders

Each decoder maps a population response $x$ to an unnormalised log-likelihood $L(c\mid x)$ for every coherence $c$.

| decoder | $L(c\mid x)$ |
|---|---|
| fixed width | $-\frac{1}{2}\left(\frac{c - a^\top x - b}{w}\right)^2$ |
| rank $r$ | $B_c^\top (A x) + b_c$, with $A$ of rank $r$ |
| full | $W_c^\top x + b_c$ |

**Training.** Spike counts are z-scored with the training fold's mean and SD. The log prior $\log\pi(c)$
(how often $c$ appears in training) is added, and the loss is cross-entropy:

$$\text{loss} = -\frac{1}{N}\sum_t \log \frac{\exp\big(L(c_t\mid x_t) + \log\pi(c_t)\big)}{\sum_{c'} \exp\big(L(c'\mid x_t) + \log\pi(c')\big)}$$

Adam (lr 0.01, weight decay 1e-4), 1,000 full-batch steps, 5-fold cross-validation. At test time the prior is
dropped, so the held-out output is a normalised likelihood $p_t(c) \propto \exp L(c\mid x_t)$.

**Loadings.** Fit on all trials, take the weight matrix $W$ (one row $W_c$ per coherence), centre it and run SVD:

$$\tilde W = W - \text{mean row}, \qquad \tilde W = U S V^\top, \qquad \text{loading}_k(c) = \tilde W_c \cdot V_k = U_{ck} S_k$$

Signs are flipped so +51.2% loads positive, and each axis is scaled to a maximum of 1 per session.

In [ ]:
class FixedWidth(nn.Module):
    """A Gaussian bump of one fixed width; only its centre moves."""
    def __init__(self, n_units, coh_values):
        super().__init__()
        self.coh = torch.tensor(coh_values, dtype=torch.float32)
        self.centre = nn.Linear(n_units, 1)
        self.log_width = nn.Parameter(torch.tensor(np.log(0.2), dtype=torch.float32))

    def forward(self, x):
        return -0.5 * ((self.coh - self.centre(x)) / torch.exp(self.log_width)) ** 2

    def weights(self):
        # only the c * (a.x) / w^2 term mixes coherence and neurons
        a = self.centre.weight.detach().numpy()[0]
        return np.outer(self.coh.numpy(), a) / torch.exp(self.log_width).item() ** 2


class LowRank(nn.Module):
    """Every coherence is read out from the same r population axes."""
    def __init__(self, n_units, n_coh, rank):
        super().__init__()
        self.A = nn.Linear(n_units, rank, bias=False)
        self.B = nn.Linear(rank, n_coh)

    def forward(self, x):
        return self.B(self.A(x))

    def weights(self):
        return self.B.weight.detach().numpy() @ self.A.weight.detach().numpy()


class Full(nn.Module):
    """Separate weights for every coherence."""
    def __init__(self, n_units, n_coh):
        super().__init__()
        self.W = nn.Linear(n_units, n_coh)

    def forward(self, x):
        return self.W(x)

    def weights(self):
        return self.W.weight.detach().numpy()


def make_model(name, n_units, coh_values):
    if name == "fixed width":
        return FixedWidth(n_units, coh_values)
    if name == "full":
        return Full(n_units, len(coh_values))
    return LowRank(n_units, len(coh_values), rank=int(name[-1]))


def train(model, x, y, steps=1000):
    log_prior = torch.log(torch.bincount(y, minlength=model(x[:1]).shape[1]) / len(y))
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
    for _ in range(steps):
        opt.zero_grad()
        loss = -torch.log_softmax(model(x) + log_prior, dim=1)[torch.arange(len(y)), y].mean()
        loss.backward()
        opt.step()
    return model


def zscore(x, mean, std):
    return torch.tensor((x - mean) / std, dtype=torch.float32)


def decode(counts, coh, coh_values):
    """Held-out log-likelihoods (trials x coherences) for each decoder, 5-fold, no prior."""
    y = np.searchsorted(coh_values, coh)
    folds = list(KFold(5, shuffle=True, random_state=0).split(counts))
    torch.manual_seed(0)
    lls = {}
    for name in MODELS:
        lls[name] = np.zeros((len(y), len(coh_values)))
        for tr, te in folds:
            mean, std = counts[tr].mean(axis=0), counts[tr].std(axis=0)
            model = train(make_model(name, counts.shape[1], coh_values),
                          zscore(counts[tr], mean, std), torch.tensor(y[tr]))
            with torch.no_grad():
                lls[name][te] = torch.log_softmax(model(zscore(counts[te], mean, std)), dim=1).numpy()
    return lls


def loadings(counts, coh, coh_values):
    """Fit each decoder on all trials; loadings = U S from the SVD of its centred weights."""
    x = zscore(counts, counts.mean(axis=0), counts.std(axis=0))
    y = torch.tensor(np.searchsorted(coh_values, coh))
    torch.manual_seed(0)
    out = {}
    for name in MODELS:
        W = train(make_model(name, counts.shape[1], coh_values), x, y).weights()
        U, S, _ = np.linalg.svd(W - W.mean(axis=0), full_matrices=False)
        k = {"fixed width": 1, "rank 1": 1, "rank 2": 2, "full": 3}[name]
        load = U[:, :k] * S[:k]
        load *= np.where(load[-1] < 0, -1, 1)
        out[name] = load / np.abs(load).max(axis=0)
    return out

## Firing over time

Each unit's mean rate per |coherence| in its preferred direction, aligned to motion onset. Only used for
Figure 1, to show why the window starts 80 ms after onset and ends before fast trials finish.

In [ ]:
BIN = 0.02
FROM_ONSET = np.arange(-0.2, 0.8 + 1e-9, BIN)


def firing_curves(task, spikes):
    """units x |coherence| x time bins, preferred direction only."""
    on, off = task["motion_on"].to_numpy(np.int64), task["motion_off"].to_numpy(np.int64)
    right = task["Direction"].astype(float).to_numpy() == 0
    strength = task["coherence"].abs().to_numpy()
    levels = np.unique(strength)

    after = on[:, None] + np.round(FROM_ONSET[:-1] * FS) >= off[:, None]       # dots already off
    in_window = (FROM_ONSET[:-1] >= WINDOW[0]) & (FROM_ONSET[:-1] < WINDOW[1])
    top = strength == levels.max()

    edges = on[:, None] + np.round(FROM_ONSET * FS).astype(np.int64)

    curves = []
    for s in spikes:
        rate = np.diff(np.searchsorted(s, edges), axis=1) / BIN
        rate[after] = np.nan
        right_is_pref = np.nanmean(rate[top & right][:, in_window]) >= np.nanmean(rate[top & ~right][:, in_window])
        pref = right if right_is_pref else ~right
        rows = [(strength == 0) if lv == 0 else (pref & (strength == lv)) for lv in levels]
        curves.append([np.nanmean(rate[r], axis=0) for r in rows])
    return levels, curves

## Run the decoders (slow, saved per session)

In [ ]:
def run_session(folder):
    rec = find(folder, "Record Node */experiment*/recording*")
    task, spikes = load_task(folder, rec), load_units(rec)
    if len(spikes) < MIN_UNITS:
        raise ValueError(f"only {len(spikes)} units")
    levels, curves = firing_curves(task, spikes)

    t, counts = spike_counts(task, spikes)
    if len(t) < MIN_TRIALS:
        raise ValueError(f"only {len(t)} trials")
    coh = t["coherence"].to_numpy(dtype=float)
    coh_values = np.unique(coh)
    lls = decode(counts, coh, coh_values)
    return {"levels": levels, "curves": curves,
            "all": {"coh": coh, "coh_values": coh_values, "n_units": counts.shape[1],
                    "choice": t["choice"].to_numpy(dtype=int), "wager": t["wager"].to_numpy(dtype=int)},
            "score": {m: lls[m][np.arange(len(coh)), np.searchsorted(coh_values, coh)] for m in MODELS},
            "lls": lls,
            "loadings": loadings(counts, coh, coh_values)}


sessions = {}
for folder in sorted(f for f in DATA.iterdir() if f.is_dir() and not f.name.startswith("@")):
    saved = Path("results") / f"{folder.name}.pkl"
    if saved.exists():
        sessions[folder.name] = pickle.loads(saved.read_bytes())
        continue
    try:
        res = run_session(folder)
    except (ValueError, KeyError) as problem:
        print(f"{folder.name}: skipped ({problem})")
        continue
    saved.write_bytes(pickle.dumps(res))
    sessions[folder.name] = res
    print(f"{folder.name}: done")

names = sorted(sessions)
print(f"{len(names)} sessions, {sum(sessions[s]['all']['n_units'] for s in names)} units, "
      f"{sum(len(sessions[s]['all']['coh']) for s in names)} trials")

## Choice and wager (fast, rerun freely)

From each trial's likelihood $p_t(c)$, the 0% coherence split half to each side:

$$R_t = \sum_{c>0} p_t(c) + \tfrac12 p_t(0), \qquad L_t = \sum_{c<0} p_t(c) + \tfrac12 p_t(0), \qquad \ell_t = \log\frac{R_t}{L_t}$$

$$H_t = -\sum_c p_t(c)\log_2 p_t(c)$$

$$P(\text{right}) = \sigma(\beta_0 + \beta_1 \ell_t), \qquad P(\text{high bet}) = \sigma(\beta_0 + \beta_1|\ell_t| + \beta_2 H_t)$$

Predictive power is the held-out gain in log-likelihood of what the monkey did over a reference model, which is either
chance (intercept only) or the stimulus (a one-hot code of coherence, $\gamma$):

$$\Delta = \frac{1}{N}\sum_t \Big[\log P_{\text{ref}+\ell}(y_t) - \log P_{\text{ref}}(y_t)\Big]$$

Logistic regression, C = 100, 10-fold stratified cross-validation.

In [ ]:
def likelihood(ll):
    return np.exp(ll - np.logaddexp.reduce(ll, axis=1, keepdims=True))


def log_odds_right(ll, coh_values):
    p = likelihood(ll)
    zero = p[:, coh_values == 0].sum(axis=1) / 2
    return np.log(p[:, coh_values > 0].sum(axis=1) + zero) - np.log(p[:, coh_values < 0].sum(axis=1) + zero)


def entropy(ll):
    p = likelihood(ll)
    return -np.sum(p * np.log2(p + 1e-12), axis=1)


def heldout_ll(features, target):
    """Log probability of what the monkey did, from a 10-fold logistic regression."""
    folds = StratifiedKFold(10, shuffle=True, random_state=0)
    p = cross_val_predict(LogisticRegression(C=100, max_iter=2000), features, target,
                          cv=folds, method="predict_proba")
    return np.log(p[np.arange(len(target)), np.searchsorted(np.unique(target), target)])


def predictive_power(res):
    a = res["all"]
    coh, choice, wager = a["coh"], a["choice"], a["wager"]
    references = {"": np.zeros((len(coh), 1)),
                  " | stimulus": (coh[:, None] == a["coh_values"]).astype(float)}
    gains = {}
    for tag, ref in references.items():
        base_choice, base_wager = heldout_ll(ref, choice), heldout_ll(ref, wager)
        for m in MODELS:
            l, H = log_odds_right(res["lls"][m], a["coh_values"]), entropy(res["lls"][m])
            gains["choice" + tag, m] = np.mean(heldout_ll(np.column_stack([ref, l]), choice) - base_choice)
            gains["wager" + tag, m] = np.mean(heldout_ll(np.column_stack([ref, np.abs(l), H]), wager) - base_wager)
    return gains


power = {s: predictive_power(sessions[s]) for s in names}

## Figures

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "normal", "legend.frameon": False,
                     "savefig.dpi": 200, "savefig.bbox": "tight"})
COLORS = {"fixed width": "tab:blue", "rank 1": "tab:orange", "rank 2": "tab:green", "full": "tab:red"}


def sem(v, axis=0):
    v = np.asarray(v)
    return v.std(axis=axis, ddof=1) / np.sqrt(v.shape[axis])


def dot_plot(ax, values, title, ylabel):
    """Faint dot per session, square = mean ± SEM."""
    for i, m in enumerate(MODELS):
        v = np.array(values[m])
        jitter = np.random.default_rng(i).uniform(-0.1, 0.1, len(v))
        ax.plot(i + jitter, v, "o", color=COLORS[m], alpha=0.35, markersize=4)
        ax.errorbar(i, v.mean(), yerr=sem(v), fmt="s", color=COLORS[m], markersize=6, capsize=3)
    ax.axhline(0, color="0.6", lw=0.6)
    ax.set_xticks(range(len(MODELS)), MODELS)
    ax.set_title(title)
    ax.set_ylabel(ylabel)

### Figure 1: firing over time
Gray shading is the decoding window. A curve stops once all of its trials have ended.

In [ ]:
levels = sessions[names[0]]["levels"]
curves = np.nanmean([unit for s in names for unit in sessions[s]["curves"]], axis=0)
shades = plt.cm.viridis(np.linspace(0, 0.9, len(levels)))

fig, ax = plt.subplots(figsize=(5.5, 3.5))
for i, lv in enumerate(levels):
    ax.plot(FROM_ONSET[:-1] + BIN / 2, curves[i], color=shades[i], lw=1.2, label=f"{lv * 100:g}%")
ax.axvline(0, color="0.6", lw=0.6)
ax.axvspan(*WINDOW, color="0.92", zorder=0)
ax.set_xlabel("time from motion onset (s)")
ax.set_ylabel("spikes/s")
ax.set_title("Firing rate, preferred direction")
ax.legend(title="|coherence|", fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig("figures/1_firing_over_time.png")

### Figure 2: uncertainty
Entropy of the held-out likelihood, mean ± SEM over sessions.

In [ ]:
strengths = np.unique(np.abs(sessions[names[0]]["all"]["coh_values"]))
fig, ax = plt.subplots(figsize=(4.5, 3.5))
for m in MODELS:
    per_session = []
    for s in names:
        H, strength = entropy(sessions[s]["lls"][m]), np.abs(sessions[s]["all"]["coh"])
        per_session.append([H[strength == lv].mean() for lv in strengths])
    ax.errorbar(strengths * 100, np.mean(per_session, axis=0), yerr=sem(per_session),
                color=COLORS[m], marker="o", markersize=3, capsize=2, label=m)
ax.set_xlabel("|coherence| (%)")
ax.set_ylabel("entropy (bits)")
ax.set_title("Uncertainty")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("figures/2_uncertainty.png")

### Figure 3: loadings
Mean ± SEM over sessions.

In [ ]:
coh_values = sessions[names[0]]["all"]["coh_values"]
fig, panels = plt.subplots(1, len(MODELS), figsize=(13, 3), sharey=True)
for ax, m in zip(panels, MODELS):
    load = np.array([sessions[s]["loadings"][m] for s in names])
    for k in range(load.shape[2]):
        ax.errorbar(coh_values * 100, load[:, :, k].mean(axis=0), yerr=sem(load[:, :, k]),
                    color=f"C{k}", marker="o", markersize=3, capsize=2, label=f"axis {k + 1}")
    ax.axhline(0, color="0.6", lw=0.6)
    ax.set_title(m.capitalize())
    ax.set_xlabel("coherence (%)")
panels[0].set_ylabel("loading")
panels[-1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig("figures/3_loadings.png")

### Figure 4: decoding the stimulus
Held-out log-likelihood of the true coherence over chance, $\frac{1}{N}\sum_t \log p_t(c_t) - \log\frac{1}{K}$.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
dot_plot(ax, {m: [sessions[s]["score"][m].mean() + np.log(len(sessions[s]["all"]["coh_values"])) for s in names]
              for m in MODELS}, "Decoding the stimulus", "relative log likelihood")
fig.tight_layout()
fig.savefig("figures/4_decoding.png")

### Figure 5: predictive power
Top: over chance. Bottom: beyond the stimulus.

In [ ]:
PANELS = [("choice", "Choice"), ("wager", "Wager"),
          ("choice | stimulus", "Choice beyond stimulus"), ("wager | stimulus", "Wager beyond stimulus")]
fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharey=True)
for ax, (key, title) in zip(axes.flat, PANELS):
    dot_plot(ax, {m: [power[s][key, m] for s in names] for m in MODELS}, title, "relative log likelihood")
for ax in axes[:, 1]:
    ax.set_ylabel("")
fig.tight_layout()
fig.savefig("figures/5_predictive_power.png")

## Numbers
Mean ± SEM over sessions, how many sessions are above 0, and a one-sample t-test against 0.

In [ ]:
def summary(label, values):
    v = np.array(values)
    print(f"  {label:22s} {v.mean():+.4f} ± {sem(v):.4f}   {np.sum(v > 0)}/{len(v)} > 0   p = {ttest_1samp(v, 0).pvalue:.2g}")


print("decoding the stimulus, difference in nats per trial:")
for a, b in [("rank 2", "rank 1"), ("rank 2", "fixed width"), ("rank 2", "full")]:
    summary(f"{a} - {b}", [sessions[s]["score"][a].mean() - sessions[s]["score"][b].mean() for s in names])

for key, _ in PANELS:
    print(f"\npredicting {key}, nats per trial:")
    for m in MODELS:
        summary(m, [power[s][key, m] for s in names])